## Aggregations

1. Создать подключение к базе данных с помощью sqlite3 библиотеки.

In [49]:
import pandas as pd
import sqlite3

conn = sqlite3.connect('../data/checking-logs.sqlite')

2. Получить схему таблицы test.

In [50]:
test_schema = pd.read_sql('PRAGMA table_info(test)', conn)

test_schema

,cid,name,type,notnull,dflt_value,pk
0,0,uid,TEXT,0,None,0
1,1,labname,TEXT,0,None,0
2,2,first_commit_ts,TIMESTAMP,0,None,0
3,3,first_view_ts,TIMESTAMP,0,None,0


3. Первые десять строк таблицы test.

In [51]:
test = pd.read_sql('SELECT * FROM test', conn)

test.head(10)

,uid,labname,first_commit_ts,first_view_ts
0,user_1,laba04,2020-04-26 17:06:18.462708,2020-04-26 21:53:59.624136
1,user_1,laba04s,2020-04-26 17:12:11.843671,2020-04-26 21:53:59.624136
2,user_1,laba05,2020-05-02 19:15:18.540185,2020-04-26 21:53:59.624136
3,user_1,laba06,2020-05-17 16:26:35.268534,2020-04-26 21:53:59.624136
4,user_1,laba06s,2020-05-20 12:23:37.289724,2020-04-26 21:53:59.624136
5,user_1,project1,2020-05-14 20:56:08.898880,2020-04-26 21:53:59.624136
6,user_10,laba04,2020-04-25 08:24:52.696624,2020-04-18 12:19:50.182714
7,user_10,laba04s,2020-04-25 08:37:54.604222,2020-04-18 12:19:50.182714
8,user_10,laba05,2020-05-01 19:27:26.063245,2020-04-18 12:19:50.182714
9,user_10,laba06,2020-05-19 11:39:28.885637,2020-04-18 12:19:50.182714


4. Найти min дельты между first_commit_ts и deadlines.

In [52]:
query_min = """
SELECT t.uid,
        MIN((julianday(t.first_commit_ts) - julianday(datetime(d.deadlines, 'unixepoch'))) * 24) AS delta_hours
FROM test as t
JOIN deadlines AS d ON LOWER(t.labname) = LOWER(d.labs)
WHERE t.labname != 'project1'
GROUP BY t.uid
ORDER BY delta_hours ASC
LIMIT 1;
"""

df_min = pd.read_sql(query_min, conn)

df_min


,uid,delta_hours
0,user_30,-202.38473


5. Найти max дельты между first_commit_ts и deadlines.

In [53]:
query_max = """
SELECT t.uid,
        MAX((julianday(t.first_commit_ts) - julianday(datetime(d.deadlines, 'unixepoch'))) * 24) AS delta_hours
FROM test as t
JOIN deadlines AS d ON LOWER(t.labname) = LOWER(d.labs)
WHERE t.labname != 'project1'
GROUP BY t.uid
ORDER BY delta_hours DESC
LIMIT 1;
"""

df_max = pd.read_sql(query_max, conn)

df_max

,uid,delta_hours
0,user_25,-2.867236


6. Найти avg дельты между first_commit_ts и deadlines (без столбца uid).

In [54]:
query_avg = """
SELECT AVG((julianday(t.first_commit_ts) - julianday(datetime(d.deadlines, 'unixepoch'))) * 24) AS delta_hours
FROM test as t
JOIN deadlines AS d ON LOWER(t.labname) = LOWER(d.labs)
WHERE t.labname != 'project1';"""

df_avg = pd.read_sql(query_avg, conn)

df_avg

,delta_hours
0,-89.687686


7. Расчитать коэффициент корреляции между pageviews и делтой.

In [55]:
query_corr = """
SELECT t.uid,
       AVG((julianday(t.first_commit_ts) - julianday(datetime(d.deadlines, 'unixepoch'))) * 24) AS avg_diff,
       COUNT(p.datetime) AS pageviews
FROM test as t
JOIN deadlines AS d ON LOWER(t.labname) = LOWER(d.labs)
LEFT JOIN pageviews AS p ON t.uid = p.uid
WHERE t.labname != 'project1'
GROUP BY t.uid;"""

views_diff = pd.read_sql(query_corr, conn)

views_diff

,uid,avg_diff,pageviews
0,user_1,-65.119644,140
1,user_10,-75.242310,445
2,user_14,-159.568696,429
3,user_17,-62.207513,235
4,user_18,-6.367907,9
5,user_19,-99.440298,64
6,user_21,-96.111041,40
7,user_25,-93.474751,895
8,user_28,-86.793652,745
9,user_3,-105.738041,1585


9. Закрыть соединение.

In [56]:
conn.close()

correlation = views_diff.corr(numeric_only=True)

correlation

,avg_diff,pageviews
avg_diff,1.000000,-0.185042
pageviews,-0.185042,1.000000
